In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from pathlib import Path

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, f1_score, accuracy_score



In [ ]:
DATA = Path("../data")
FIG = Path("../figures")
FIG.mkdir(parents=True, exist_ok=True)



In [ ]:
# ======================
# 1. ЗАГРУЗКА ДАННЫХ
# ======================
df = pd.read_csv(DATA / "fetal_sample.csv")



In [ ]:
print("Number of samples:", df.shape[0])
print("Columns:", list(df.columns))



In [ ]:
df.info()
df.head()



In [ ]:
df.describe().T



In [ ]:
# Подготовка данных для визуализации баланса классов
class_counts = df["target"].value_counts().sort_values(ascending=True)
colors = sns.color_palette("viridis", len(class_counts))
percentages = (class_counts / class_counts.sum()) * 100

plt.figure(figsize=(8, 6))
ax = plt.subplot(111)

# Круговая диаграмма (donut) для классов CTG / NSP-подобных меток
wedges, texts = ax.pie(
    class_counts,
    startangle=140,
    colors=colors,
    wedgeprops=dict(width=0.5, edgecolor="w", linewidth=0.5)  # Отступы между секторами
)

# Настройка легенды с числом и долей класса
legend_labels = [
    f"{label} ({count:,} - {pct:.1f}%)"
    for label, count, pct in zip(class_counts.index, class_counts, percentages)
]
plt.legend(
    wedges,
    legend_labels,
    title="Fetal classes",
    loc="center left",
    bbox_to_anchor=(1, 0.5),
    fontsize=10,
)

plt.title("Distribution of fetal CTG classes", pad=20, fontsize=14, fontweight="bold")

# Белый круг в центре + общее число наблюдений
centre_circle = plt.Circle((0, 0), 0.3, color="white")
ax.add_artist(centre_circle)
plt.text(0, 0, f"Total:\n{sum(class_counts):,}", ha="center", va="center", fontsize=12)

plt.tight_layout()
plt.savefig(FIG / "fetal_class_counts.png", dpi=120, bbox_inches="tight")
plt.show()



In [ ]:
# Установка размера фигуры (10 дюймов по ширине, 5 по высоте)
plt.figure(figsize=(10, 5))

# Столбчатая диаграмма баланса классов
sns.barplot(
    x=class_counts.index,   # Категории NSP-подобных меток
    y=class_counts.values,  # Число образцов в каждом классе
    hue=class_counts.index,
    palette="viridis",
    legend=False,
)

plt.title("Class counts (fetal CTG sample)", fontsize=16, pad=16)
plt.xlabel("Target class", fontsize=12)
plt.ylabel("Number of samples", fontsize=12)
plt.grid(axis="y", linestyle="--", alpha=0.7)

# Аннотации над столбцами
for i, count in enumerate(class_counts.values):
    plt.text(i, count + 5, str(count), ha="center", va="bottom", fontsize=10)

plt.tight_layout()
plt.show()



In [ ]:
# Тепловая карта корреляции числовых признаков
num_cols = df.select_dtypes(include=[np.number]).columns
corr = df[num_cols].corr()

plt.figure(figsize=(12, 9))
sns.heatmap(
    corr,
    cmap="coolwarm",
    annot=False,          # Без чисел — признаков много, иначе шум
    square=True,
    linewidths=0.3,
    cbar_kws={"shrink": 0.8},
)
plt.title("Correlation between numeric features", fontsize=14, pad=12)
plt.tight_layout()
plt.show()



In [ ]:
# Гистограммы признаков — смотрим формы распределений и выбросы
plot_cols = [c for c in df.columns if c != "target"][:8]
df[plot_cols].hist(figsize=(14, 10), bins=25, edgecolor="black", color="#4c72b0")
plt.suptitle("Feature histograms (first 8 columns)", fontsize=16)
plt.tight_layout()
plt.show()



In [ ]:
# ======================
# 2. ПОДГОТОВКА ПРИЗНАКОВ
# ======================
X = df.drop(columns=["target"])
y_raw = df["target"]

# Кодируем текстовые метки в целые классы 0..K-1
le = LabelEncoder()
y = le.fit_transform(y_raw)
print("Classes:", list(le.classes_))
print("X shape:", X.shape)



In [ ]:
# Разделение на train/test со стратификацией по классам
RANDOM_SEED = 42
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_SEED, stratify=y
)
print("Train:", X_train.shape[0], "Test:", X_test.shape[0])



In [ ]:
# Нормализация признаков (важно для нейросети и чувствительных моделей)
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)



In [ ]:
# ======================
# 3. SKLEARN BASELINE (RandomForest)
# ======================
rf = RandomForestClassifier(
    n_estimators=200,
    max_depth=12,
    random_state=RANDOM_SEED,
    n_jobs=-1,
)
rf.fit(X_train_s, y_train)
rf_pred = rf.predict(X_test_s)
print("RF accuracy:", round(accuracy_score(y_test, rf_pred), 4))
print("RF F1 macro:", round(f1_score(y_test, rf_pred, average="macro"), 4))
print(classification_report(y_test, rf_pred, target_names=le.classes_))



In [ ]:
# ======================
# 4. KERAS MLP
# ======================
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

num_classes = len(le.classes_)
y_train_oh = keras.utils.to_categorical(y_train, num_classes=num_classes)
y_test_oh = keras.utils.to_categorical(y_test, num_classes=num_classes)

model = keras.Sequential([
    layers.Input(shape=(X_train_s.shape[1],)),
    layers.Dense(128, activation="relu"),
    layers.Dropout(0.3),
    layers.Dense(64, activation="relu"),
    layers.Dropout(0.3),
    layers.Dense(32, activation="relu"),
    layers.Dense(num_classes, activation="softmax"),
])

model.compile(
    optimizer="adam",
    loss="categorical_crossentropy",
    metrics=["accuracy"],
)
model.summary()



In [ ]:
# ======================
# 5. ОБУЧЕНИЕ МОДЕЛИ
# ======================
EPOCHS = 40
BATCH_SIZE = 32

history = model.fit(
    X_train_s,
    y_train_oh,
    validation_data=(X_test_s, y_test_oh),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    verbose=1,
)



In [ ]:
# ======================
# 6. МЕТРИКИ
# ======================
y_prob = model.predict(X_test_s)
y_pred = np.argmax(y_prob, axis=1)

print("MLP accuracy:", round(accuracy_score(y_test, y_pred), 4))
print("MLP F1 macro:", round(f1_score(y_test, y_pred, average="macro"), 4))
print("MLP F1 micro:", round(f1_score(y_test, y_pred, average="micro"), 4))
print(classification_report(y_test, y_pred, target_names=le.classes_))



In [ ]:
# ======================
# 7. ВИЗУАЛИЗАЦИЯ ОБУЧЕНИЯ
# ======================
plt.figure(figsize=(12, 5))

# График точности
plt.subplot(1, 2, 1)
plt.plot(history.history["accuracy"], label="Train Accuracy", color="blue")
plt.plot(history.history["val_accuracy"], label="Validation Accuracy", color="red")
plt.title("Training and Validation Accuracy")
plt.xlabel("Epochs")
plt.ylabel("Accuracy")
plt.legend()
plt.grid(alpha=0.3)

# График потерь
plt.subplot(1, 2, 2)
plt.plot(history.history["loss"], label="Train Loss", color="blue")
plt.plot(history.history["val_loss"], label="Validation Loss", color="red")
plt.title("Training and Validation Loss")
plt.xlabel("Epochs")
plt.ylabel("Loss")
plt.legend()
plt.grid(alpha=0.3)

plt.tight_layout()
plt.show()



In [ ]:
# Матрица ошибок MLP
cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(7, 6))
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=le.classes_,
    yticklabels=le.classes_,
)
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Confusion Matrix (MLP)")
plt.tight_layout()
plt.show()



In [ ]:
# Важность признаков из RandomForest (для интерпретации CTG-признаков)
importances = pd.Series(rf.feature_importances_, index=X.columns).sort_values(ascending=True)

plt.figure(figsize=(8, 6))
importances.plot(kind="barh", color=sns.color_palette("viridis", len(importances)))
plt.title("RandomForest feature importance", fontsize=14, pad=12)
plt.xlabel("Importance")
plt.tight_layout()
plt.show()

